# Distributed training  

### Import packages 
* os -  provides a portable way of using operating system dependent functionality.

In [1]:
import os

### Importing standard python packages
* utils - a collection of small Python functions and classes which make common patterns shorter and easier.
* numpy - package for scientific computing with Python.

In [2]:
import utils
import numpy as np

/home/ec2-user/anaconda3/envs/tensorflow_p27/lib/python2.7/site-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters


### Importing tensorflow packages
* tensorflow - library for dataflow programming across a range of tasks.
* tensorflow.contrib.learn.python.learn.datasets - contains dataset utilities and synthetic/reference datasets, for getting the mnist dataset

In [48]:
import tensorflow as tf
from tensorflow.examples.tutorials.mnist import input_data
from tensorflow.contrib.learn.python.learn.datasets import mnist

### Importing amazon packages
* sagemaker - Python SDK for training and deploying machine learning models on Amazon SageMaker.
* get_execution_role - Return the role ARN whose credentials are used to call the API.
* sagemaker.tensorflow - The Amazon SageMaker custom TensorFlow code. 

In [49]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.tensorflow import TensorFlow

## Getting and preprocessing the dataset
* Read the mnist dataset
* split it into three : train, validation and test
* instantiate a sagemaker session
* upload our datasets to an S3 location.

In [50]:
data_sets = mnist.read_data_sets('data', dtype=tf.uint8, reshape=False, validation_size=5000)

Extracting data/train-images-idx3-ubyte.gz
Extracting data/train-labels-idx1-ubyte.gz
Extracting data/t10k-images-idx3-ubyte.gz
Extracting data/t10k-labels-idx1-ubyte.gz


In [51]:
utils.convert_to(data_sets.train, 'train', 'data')
utils.convert_to(data_sets.validation, 'validation', 'data')
utils.convert_to(data_sets.test, 'test', 'data')

('Writing', 'data/train.tfrecords')
('Writing', 'data/validation.tfrecords')
('Writing', 'data/test.tfrecords')


In [52]:
sagemaker_session = sagemaker.Session()

In [53]:
inputs = sagemaker_session.upload_data(path='data', key_prefix='data/mnist')

INFO:sagemaker:Created S3 bucket: sagemaker-us-east-1-771620814942


## Training the model
* You will need the training script predefined.
* Define a tensorflow estimator object and pass in the python script as the entry point parameter.
* Your TensorFlow training script must be a Python 2.7 source file. It should define the following methods.
* **model_fn**: Defines the model that will be trained.
* **train_input_fn**: Preprocess and load training data.
* **eval_input_fn**: Preprocess and load evaluation data.
* **serving_input_fn**: Defines the features to be passed to the model during prediction.
* Get the role ARN whose credentials are used to call the API.
* To perform distributed training,the instance count is set to 2. 
* Invoke the fit method to train the model. The fit method will create a training job in two ml.c4.xlarge instances. The logs will show the instances doing training, evaluation, and incrementing the number of training steps.
* Invoke the deploy method to create an endpoint.

In [54]:
!cat 'mnist.py'

import os
import tensorflow as tf
from tensorflow.python.estimator.model_fn import ModeKeys as Modes

INPUT_TENSOR_NAME = 'inputs'
SIGNATURE_NAME = 'predictions'

LEARNING_RATE = 0.001


def model_fn(features, labels, mode, params):
    # Input Layer
    input_layer = tf.reshape(features[INPUT_TENSOR_NAME], [-1, 28, 28, 1])

    # Convolutional Layer #1
    conv1 = tf.layers.conv2d(
        inputs=input_layer,
        filters=32,
        kernel_size=[5, 5],
        padding='same',
        activation=tf.nn.relu)

    # Pooling Layer #1
    pool1 = tf.layers.max_pooling2d(inputs=conv1, pool_size=[2, 2], strides=2)

    # Convolutional Layer #2 and Pooling Layer #2
    conv2 = tf.layers.conv2d(
        inputs=pool1,
        filters=64,
        kernel_size=[5, 5],
        padding='same',
        activation=tf.nn.relu)
    pool2 = tf.layers.max_pooling2d(inputs=conv2, pool_size=[2, 2], strides=2)

    # Dense Layer
    pool2_flat = tf.reshape(pool2, [-1, 7 * 7 * 64])
    dense = tf.layers.d

In [55]:
role = get_execution_role()
role

'arn:aws:iam::771620814942:role/service-role/AmazonSageMaker-ExecutionRole-20171212T090635'

In [56]:
mnist_estimator = TensorFlow(entry_point='mnist.py',
                             role=role,
                             training_steps=1000, 
                             evaluation_steps=100,
                             train_instance_count=2,
                             train_instance_type='ml.c4.xlarge')

In [57]:
mnist_estimator.fit(inputs)

INFO:sagemaker:Created S3 bucket: sagemaker-us-east-1-771620814942
INFO:sagemaker:Creating training-job with name: sagemaker-tensorflow-2018-04-29-18-41-19-646


...........................................
2018-04-29 18:44:45,347 INFO - root - running container entrypoint
2018-04-29 18:44:45,348 INFO - root - starting train task
2018-04-29 18:44:45,353 INFO - container_support.training - Training starting
/usr/local/lib/python2.7/dist-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
2018-04-29 18:44:47,058 INFO - botocore.vendored.requests.packages.urllib3.connectionpool - Starting new HTTP connection (1): 169.254.170.2
2018-04-29 18:44:47,253 INFO - botocore.vendored.requests.packages.urllib3.connectionpool - Starting new HTTPS connection (1): sagemaker-us-east-1-771620814942.s3.amazonaws.com
2018-04-29 18:44:47,383 INFO - botocore.vendored.requests.packages.urllib3.connectionpool - Starting new HTTPS connection (1): s3.am

2018-04-29 18:45:13.050550: I tensorflow/core/distributed_runtime/master.cc:221] CreateSession still waiting for response from worker: /job:master/replica:0/task:0
2018-04-29 18:45:13.050581: I tensorflow/core/distributed_runtime/master.cc:221] CreateSession still waiting for response from worker: /job:ps/replica:0/task:0
2018-04-29 18:45:23.050775: I tensorflow/core/distributed_runtime/master.cc:221] CreateSession still waiting for response from worker: /job:master/replica:0/task:0
2018-04-29 18:45:23.050807: I tensorflow/core/distributed_runtime/master.cc:221] CreateSession still waiting for response from worker: /job:ps/replica:0/task:0
2018-04-29 18:45:33.050982: I tensorflow/core/distributed_runtime/master.cc:221] CreateSession still waiting for response from worker: /job:master/replica:0/task:0
2018-04-29 18:45:33.051042: I tensorflow/core/distributed_runtime/master.cc:221] CreateSession still waiting for response from worker: /job:ps/replica:0/task:0
2018-04-29 18:45:43.051228: 

2018-04-29 18:47:11,448 INFO - tensorflow - Saving checkpoints for 1 into s3://sagemaker-us-east-1-771620814942/sagemaker-tensorflow-2018-04-29-18-41-19-646/checkpoints/model.ckpt.
2018-04-29 18:47:11.493141: I tensorflow/core/platform/s3/aws_logging.cc:54] Connection has been released. Continuing.
2018-04-29 18:47:11.503779: I tensorflow/core/platform/s3/aws_logging.cc:54] Deleting file: /tmp/s3_filesystem_XXXXXX20180429T1847111525027631492
2018-04-29 18:47:11.507050: I tensorflow/core/platform/s3/aws_logging.cc:54] Connection has been released. Continuing.
2018-04-29 18:47:11.522968: I tensorflow/core/platform/s3/aws_logging.cc:54] Deleting file: /tmp/s3_filesystem_XXXXXX20180429T1847111525027631503
2018-04-29 18:47:11.523191: I tensorflow/core/platform/s3/aws_logging.cc:54] Connection has been released. Continuing.
2018-04-29 18:47:11.536486: I tensorflow/core/platform/s3/aws_logging.cc:54] Connection has been released. Continuing.
2018-04-29 18:47:11.607768: I tensorflow/core/platf

2018-04-29 18:47:22,630 INFO - tensorflow - Evaluation [80/100]
2018-04-29 18:47:23,484 INFO - tensorflow - Evaluation [90/100]
2018-04-29 18:47:24,326 INFO - tensorflow - Evaluation [100/100]
2018-04-29 18:47:24,358 INFO - tensorflow - Finished evaluation at 2018-04-29-18:47:24
2018-04-29 18:47:24,358 INFO - tensorflow - Saving dict for global step 1: accuracy = 0.2188, global_step = 1, loss = 2.1889563
2018-04-29 18:47:24.358939: I tensorflow/core/platform/s3/aws_logging.cc:54] Connection has been released. Continuing.
2018-04-29 18:47:24.447759: E tensorflow/core/platform/s3/aws_logging.cc:60] No response body. Response code: 404
2018-04-29 18:47:24.447964: W tensorflow/core/platform/s3/aws_logging.cc:57] If the signature check failed. This could be because of a time skew. Attempting to adjust the signer.
2018-04-29 18:47:24.448298: I tensorflow/core/platform/s3/aws_logging.cc:54] Connection has been released. Continuing.
2018-04-29 18:47:24.460479: I tensorflow/core/platform/s3/aws

2018-04-29 18:47:43.053850: I tensorflow/core/distributed_runtime/master.cc:221] CreateSession still waiting for response from worker: /job:ps/replica:0/task:0
2018-04-29 18:47:50.854371: I tensorflow/core/distributed_runtime/master_session.cc:1024] Start master session 07391f454151a40f with config: allow_soft_placement: true
2018-04-29 18:47:50,931 INFO - tensorflow - Running local_init_op.
2018-04-29 18:47:50,935 INFO - tensorflow - Done running local_init_op.
2018-04-29 18:47:51,626 INFO - tensorflow - loss = 0.16930065, step = 61
2018-04-29 18:47:59,333 INFO - tensorflow - global_step/sec: 3.22007
2018-04-29 18:48:06,841 INFO - tensorflow - loss = 0.086431, step = 138 (38.872 sec)
2018-04-29 18:48:19,299 INFO - tensorflow - global_step/sec: 5.05851
2018-04-29 18:48:31,937 INFO - tensorflow - loss = 0.040605787, step = 264 (40.311 sec)
2018-04-29 18:48:39,442 INFO - tensorflow - global_step/sec: 5.0141
2018-04-29 18:48:45,139 INFO - tensorflow - loss = 0.05098033, step = 331 (38.298

2018-04-29 18:51:02,504 INFO - tensorflow - Evaluation [10/100]
2018-04-29 18:51:03,552 INFO - tensorflow - Evaluation [20/100]
2018-04-29 18:51:04,403 INFO - tensorflow - Evaluation [30/100]
2018-04-29 18:51:05,284 INFO - tensorflow - Evaluation [40/100]
2018-04-29 18:51:06,215 INFO - tensorflow - Evaluation [50/100]
2018-04-29 18:51:07,039 INFO - tensorflow - Evaluation [60/100]
2018-04-29 18:51:08,028 INFO - tensorflow - Evaluation [70/100]
2018-04-29 18:51:08,904 INFO - tensorflow - Evaluation [80/100]
2018-04-29 18:51:09,850 INFO - tensorflow - Evaluation [90/100]
2018-04-29 18:51:10,867 INFO - tensorflow - Evaluation [100/100]
2018-04-29 18:51:10,894 INFO - tensorflow - Finished evaluation at 2018-04-29-18:51:10
2018-04-29 18:51:10,894 INFO - tensorflow - Saving dict for global step 1002: accuracy = 0.9888, global_step = 1002, loss = 0.03568445
2018-04-29 18:51:10.894855: I tensorflow/core/platform/s3/aws_logging.cc:54] Connection has been released. Continuing.
2018-04-29 18:51:1

===== Job Complete =====
Billable seconds: 1050


In [58]:
mnist_predictor = mnist_estimator.deploy(initial_instance_count=1,
                                             instance_type='ml.m4.xlarge')

INFO:sagemaker:Creating model with name: sagemaker-tensorflow-2018-04-29-18-41-19-646
INFO:sagemaker:Creating endpoint with name sagemaker-tensorflow-2018-04-29-18-41-19-646


--------------------------------------------------------------!

## Validating the model
* get some data for testing
* call the predictor to compare the labels from test data and the predicted labels

In [59]:
mnist = input_data.read_data_sets("/tmp/data/", one_hot=True)

Extracting /tmp/data/train-images-idx3-ubyte.gz
Extracting /tmp/data/train-labels-idx1-ubyte.gz
Extracting /tmp/data/t10k-images-idx3-ubyte.gz
Extracting /tmp/data/t10k-labels-idx1-ubyte.gz


In [60]:
for i in range(10):
    data = mnist.test.images[i].tolist()
    tensor_proto = tf.make_tensor_proto(values=np.asarray(data), shape=[1, len(data)], dtype=tf.float32)
    predict_response = mnist_predictor.predict(tensor_proto)
    
    print("========================================")
    label = np.argmax(mnist.test.labels[i])
    print("label is {}".format(label))
    prediction = predict_response['outputs']['classes']['int64Val'][0]
    print("prediction is {}".format(prediction))

label is 7
prediction is 7
label is 2
prediction is 2
label is 1
prediction is 1
label is 0
prediction is 0
label is 4
prediction is 4
label is 1
prediction is 1
label is 4
prediction is 4
label is 9
prediction is 9
label is 5
prediction is 5
label is 9
prediction is 9


## Delete the endpoint

In [61]:
#sagemaker_session.delete_endpoint(mnist_predictor.endpoint)